#Generate Quetsions and Answers Json

In [1]:
# Install dependencies
!pip install nltk scikit-learn faiss-cpu bert-score transformers

# Imports
import os
import json
import time
import torch
import numpy as np
import torch.nn as nn
from collections import Counter
from sklearn.metrics.pairwise import cosine_similarity
from nltk.tokenize import word_tokenize
from nltk.translate.bleu_score import sentence_bleu, SmoothingFunction
from bert_score import score as bert_score

# Download NLTK tokenizer
import nltk
nltk.download('punkt')



   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 31.3/31.3 MB 58.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.1/61.1 kB 4.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 1.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 102.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 81.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 43.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 2.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 5.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 8.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.9/127.9 MB 7.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.5/207.5 MB 4.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 188.7/188.7 MB 5.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt.zip.


True

In [12]:
folder_path = "/content/Research-Chatbot/Training"  # Update if needed

qa_pairs_combined_raw = []

for file_name in os.listdir(folder_path):
    file_path = os.path.join(folder_path, file_name)
    if file_name.endswith('.json'):
        try:
            with open(file_path, 'r', encoding='utf-8', errors='ignore') as f:
                qa_list = json.load(f)
                if isinstance(qa_list, list):
                    qa_pairs_combined_raw.extend(qa_list)
                    print(f"Loaded {len(qa_list)} QA pairs from: {file_name}")
                else:
                    print(f"Skipping {file_name}: Not a list of QA pairs.")
        except json.JSONDecodeError:
            print(f"JSON Decode Error in file: {file_name}")
        except Exception as e:
            print(f"Error reading {file_name}: {e}")
    else:
        print(f"Skipping non-JSON file: {file_name}")

qa_pairs = qa_pairs_combined_raw


Loaded 349 QA pairs from: Training_set_v2.json
Skipping non-JSON file: .ipynb_checkpoints


In [13]:
nltk.download('punkt_tab')

[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


True

Another implementation

In [14]:
embedding_dim = 100
hidden_dim = 128
max_len = 20

# Prepare dataset
questions = [q["query"].lower() for q in qa_pairs if "query" in q]
answers = [q["expected_answer"] for q in qa_pairs if "expected_answer" in q]

# Build vocabulary
all_words = [word for q in questions for word in word_tokenize(q)]
word_freq = Counter(all_words)
vocab = {word: i + 2 for i, (word, _) in enumerate(word_freq.items())}
vocab["<pad>"] = 0
vocab["<unk>"] = 1
vocab_size = len(vocab)
print(f"Vocabulary size: {vocab_size}")

# Encoding function
def encode_question(q):
    tokens = word_tokenize(q.lower())
    idxs = [vocab.get(w, 1) for w in tokens]
    padded = idxs[:max_len] + [0] * (max_len - len(idxs))
    return torch.tensor(padded)


Vocabulary size: 642


In [15]:
class LSTMEncoder(nn.Module):
    def __init__(self, vocab_size, embedding_dim, hidden_dim):
        super(LSTMEncoder, self).__init__()
        self.embedding = nn.Embedding(vocab_size, embedding_dim)
        self.lstm = nn.LSTM(embedding_dim, hidden_dim, batch_first=True)
        self.fc = nn.Linear(hidden_dim, hidden_dim)

    def forward(self, input_ids):
        embedded = self.embedding(input_ids)
        _, (hidden, _) = self.lstm(embedded)
        return self.fc(hidden[-1])  # Return final hidden state


In [16]:
encoded_questions = torch.stack([encode_question(q) for q in questions])

model = LSTMEncoder(vocab_size=vocab_size, embedding_dim=embedding_dim, hidden_dim=hidden_dim)
model.eval()

with torch.no_grad():
    question_vecs_tensor = model(encoded_questions).float()  # final embeddings


In [17]:
import torch.nn.functional as F # Import torch.nn.functional here

def lstm_chatbot():
    print("LSTM Chatbot ready. Type 'exit' to quit.")
    while True:
        user_input = input("You: ").lower().strip()
        if user_input in ['exit', 'quit']:
            print("Chatbot: Goodbye!")
            break

        input_vec = encode_question(user_input).unsqueeze(0)
        with torch.no_grad():
            query_vec = model(input_vec).float()
            similarities = F.cosine_similarity(query_vec, question_vecs_tensor)
            best_idx = torch.argmax(similarities).item()
            best_score = similarities[best_idx].item()

        if best_score < 0.6:
            print("Chatbot: Sorry, I don't understand.")
        else:
            print("Chatbot:", answers[best_idx])

In [18]:
lstm_chatbot()


LSTM Chatbot ready. Type 'exit' to quit.
You: exit
Chatbot: Goodbye!


#Evaluation Pipeline

In [19]:
def evaluate_lstm_model(test_set, model, qa_pairs, vocab, max_len, question_vecs_tensor, threshold=0.5):
    results = []
    total_time = 0
    all_generated = []
    all_expected = []

    model.eval()
    smoothie = SmoothingFunction().method4

    for item in test_set:
        query = item["query"]
        expected = item["expected_answer"]

        start_time = time.time()
        input_vec = encode_question(query).unsqueeze(0)
        with torch.no_grad():
            query_embedding = model(input_vec).float()
            similarities = F.cosine_similarity(query_embedding, question_vecs_tensor)
            best_match_index = torch.argmax(similarities).item()
            best_score = similarities[best_match_index].item()

        if best_score >= threshold:
            generated = qa_pairs[best_match_index]["expected_answer"]
        else:
            generated = "I'm sorry, I don't know the answer to that."

        response_time = time.time() - start_time
        total_time += response_time

        # Exact Match
        exact_match = int(expected.lower() in generated.lower()) if expected else 0

        # BLEU Score
        bleu = sentence_bleu([expected.split()], generated.split(), smoothing_function=smoothie) if expected else 0.0

        results.append({
            "Query": query,
            "Generated": generated,
            "Expected": expected,
            "ExactMatch": exact_match,
            "BLEU": bleu,
            "TimeTaken": response_time
        })

        all_generated.append(generated)
        all_expected.append(expected)

    # Filter out empty expected answers for BERTScore
    filtered_generated = [g for g, e in zip(all_generated, all_expected) if e.strip()]
    filtered_expected = [e for e in all_expected if e.strip()]

    if filtered_expected:
        P, R, F1 = bert_score(filtered_generated, filtered_expected, lang="en", verbose=True)
        avg_bertscore_f1 = F1.mean().item()
    else:
        avg_bertscore_f1 = 0.0

    # Summary
    accuracy = sum(r["ExactMatch"] for r in results) / len(results)
    avg_bleu = sum(r["BLEU"] for r in results) / len(results)
    avg_time = total_time / len(results)

    print("\n--- Evaluation Summary ---")
    print(f"Accuracy (Exact Match): {accuracy:.2f}")
    print(f"Average BLEU Score: {avg_bleu:.2f}")
    print(f"Average BERTScore F1: {avg_bertscore_f1:.2f}")
    print(f"Average Response Time: {avg_time:.2f}s")

    return results


In [20]:
test_set = [
    {"query": "What is the acronym for Business School?", "expected_answer": "EU"},
    {"query": "What business school did the agreement between DBS and?", "expected_answer": "EU Business School"},
    {"query": "Where is the Business School located in Germany?", "expected_answer": "EU Business School"},
    {"query": "What ratings did DBS earned?", "expected_answer": "4 stars"},
    {"query": "how to view Library account?", "expected_answer": ""},
    {"query": "Are the Guides to Library resources for students with disabilities are also available in the Library?", "expected_answer": "on the library website"},
    {"query": "How many university partnerships does DBS has developed?", "expected_answer": "over 75"}
]

In [24]:
file_path = r"/content/Research-Chatbot/Test/Test_set_v5.0.json"

# Load test_set from file
def load_test_set(path):
    if not os.path.exists(path):
        raise FileNotFoundError(f"The file {path} does not exist.")
    with open(path, "r", encoding="utf-8") as f:
        data = json.load(f)
    return data

if __name__ == "__main__":
    try:
        test_set = load_test_set(file_path)
        print("Test set loaded successfully.")
        for item in test_set:
            print(f"Query: {item.get('query', '')}")
            print(f"Expected Answer: {item.get('expected_answer', '')}")
            print("-" * 50)
    except Exception as e:
        print(f"Error: {e}")

Test set loaded successfully.
Query: What is PSI?
Expected Answer: professional body for psychology in Ireland
--------------------------------------------------
Query: What country has the highest reputation for leading independent colleges?
Expected Answer: Ireland
--------------------------------------------------
Query: student population of DBS?
Expected Answer: over 9,000
--------------------------------------------------
Query: What is the name of Ireland's largest independent third-level college?
Expected Answer: Dublin Business School
--------------------------------------------------
Query: When was Dublin Business School established?
Expected Answer: 1975
--------------------------------------------------
Query: When was DBS established?
Expected Answer: 1975
--------------------------------------------------
Query: DBS was established in which year?
Expected Answer: 1975
--------------------------------------------------
Query: What is the name of the society that represent

In [25]:


results = evaluate_lstm_model(
    test_set=test_set,
    model=model,
    qa_pairs=qa_pairs,
    vocab=vocab,
    max_len=max_len,
    question_vecs_tensor=question_vecs_tensor,
    threshold=0.5
)



Some weights of RobertaModel were not initialized from the model checkpoint at roberta-large and are newly initialized: ['pooler.dense.bias', 'pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


calculating scores...
computing bert embedding.


  0%|          | 0/5 [00:00<?, ?it/s]

computing greedy matching.


  0%|          | 0/3 [00:00<?, ?it/s]

done in 83.30 seconds, 2.06 sentences/sec

--- Evaluation Summary ---
Accuracy (Exact Match): 0.45
Average BLEU Score: 0.21
Average BERTScore F1: 0.87
Average Response Time: 0.00s


In [26]:
results

[{'Query': 'What is PSI?',
  'Generated': 'the Department of Education',
  'Expected': 'professional body for psychology in Ireland',
  'ExactMatch': 0,
  'BLEU': 0,
  'TimeTaken': 0.0035278797149658203},
 {'Query': 'What country has the highest reputation for leading independent colleges?',
  'Generated': 'The Careers Hub offers free career support workshops, podcasts with industry professionals, and employer events.',
  'Expected': 'Ireland',
  'ExactMatch': 0,
  'BLEU': 0,
  'TimeTaken': 0.0028777122497558594},
 {'Query': 'student population of DBS?',
  'Generated': 'the Department of Education',
  'Expected': 'over 9,000',
  'ExactMatch': 0,
  'BLEU': 0,
  'TimeTaken': 0.001940011978149414},
 {'Query': "What is the name of Ireland's largest independent third-level college?",
  'Generated': 'Arts, Foreign Languages, Communication, Tourism, and Cultural Heritage.',
  'Expected': 'Dublin Business School',
  'ExactMatch': 0,
  'BLEU': 0,
  'TimeTaken': 0.002135038375854492},
 {'Query':